## Methodology

In this notebook, we evaluate how well different LLMs can identify metaphor usage in text by comparing their outputs against human-labeled data.

We use the survey dataset from *“From Tools to Thieves” (Cheng et al.)* as the reference benchmark. The dataset contains open-ended responses where participants described AI using metaphors, along with manually annotated “dominant metaphor” labels.

The metaphor categories are based on Cheng et al.’s taxonomy, summarized in the figure below.

<div style="text-align: center;">
    <img src="../Docs_Images/Metaphors_Myra_Cheng.png" width="65%">
    <p style="font-size: 12px; color: gray;">
        Figure 1: Metaphor categories adapted from Cheng et al. (2025).
    </p>
</div>

From the full set of metaphors in the paper, we restrict the evaluation to the categories that overlap exactly with the categories used in FrameScope. This avoids introducing subjective mapping decisions between taxonomies.

For each response:
- We prompt GPT and LLaMA-based models to predict the dominant metaphor.
- We compare the model prediction directly with the human-labeled metaphor.
- We evaluate model performance using standard classification metrics.

The goal is not only to measure accuracy, but to understand how reliably different models capture the way people describe AI.

| Category | Definition | Example |
|---|---|---|
| Tool | AI as a functional instrument | "A productivity tool" |
| Weapon | AI framed through conflict/war | "AI is a weapon against truth" |
| Mirror | Reflects human data or bias | "A mirror of the internet" |
| Garbage | AI output as junk/slop | "AI-generated garbage" |
| Black Box | Opaque system | "A black box" |
| Mind | Human-like thinking | "It understands me" |
| Friend | Companion/social partner | "Feels like a friend" |
| Child | Developing/immature system | "AI is still a baby" |
| Criminal | Wrongdoing/stealing | "AI steals art" |
| Artist | Creative producer | "AI creates art" |
| Assistant | Helper role | "My assistant" |
| Animal | Wild/uncontrolled entity | "A beast we unleashed" |
| Disaster | Overwhelming force | "A tsunami of AI" |
| Momentum | Unstoppable process | "Train has left the station" |
| Disease | Contagion/spread | "AI is infecting the web" |
| God | Omniscient/powerful | "God-level intelligence" |
| Genie | Wish-granting but risky | "Like a genie" |
| Oracle | Source of answers | "People treat it like an oracle" |

## Models Evaluated

We evaluate both open-source (local) and proprietary LLMs to understand how model size, architecture, and cost trade-offs affect metaphor classification performance.

### LLaMA / Open-Source Models (via Ollama)

- **llama3.1:8b** — lightweight baseline model  
- **llama3.1:70b** — large model for high-quality benchmark  
- **llama3.2** — newer LLaMA variant with improved reasoning  

- **mistral** — strong general-purpose open model  
- **mixtral** — mixture-of-experts model, often performs well on structured tasks  
- **gemma:7b** — Google model with different training distribution  
- **phi3** — small but efficient model, useful for testing lower-capacity behavior  
- **qwen2** — alternative architecture with strong reasoning performance  

**Rationale:**  
These models cover a range of sizes (small → large), architectures (dense vs MoE), and training sources. This allows us to test whether improvements in metaphor detection come from scale, architecture, or training data differences.

---

### GPT / Proprietary Models

- **gpt-5-nano, gpt-4.1-nano** — ultra-low-cost baselines  
- **gpt-5-mini, gpt-4.1-mini, gpt-4o-mini** — cost-efficient models with reasonable performance  

- **gpt-5, gpt-4o, o4-mini** — mid-tier models balancing cost and capability  
- **o3-mini** — reasoning-oriented model  

- **gpt-4.1** — high-quality benchmark model  

**Rationale:**  
These models are selected to systematically explore the trade-off between cost and performance.  
- Smaller models test how far low-cost inference can go  
- Mid-tier models capture practical deployment scenarios  
- High-end models provide an upper bound on performance  

---

### Overall Design

Across both groups, the model set is designed to answer:

- How much does model size matter for metaphor understanding?  
- Do reasoning-oriented models perform better on nuanced linguistic tasks?  
- Can cheaper models approximate the performance of larger ones?  

This setup allows for a structured comparison rather than a one-off evaluation.

# Importing the dataset

In [1]:
import pandas as pd
df = pd.read_csv('../Data/LLM_Comparison/Metaphor_Data_Myra_Chang.csv')

In [2]:
df.head(5)

,month,age,gender,ethnicity,ai_metaphor,anthroscore,competence,warmth,dominant_metaphor
0,may,49,M,White,genie in a bottle - you can ask it to do anyth...,3.115036,0.175481,0.142828,genie
1,may,27,M,White,Artificial intelligence is a tool that humans ...,-2.247446,0.051367,0.045759,tool
2,may,28,W,White,AI is a choose-your-own-adventure book where y...,-5.525272,0.184953,0.108136,library
3,may,31,M,Asian,AI is like a printer or copier machine: extrem...,-4.664262,0.214834,0.093366,machine
4,may,50,M,White,AI makes fewer medical mistakes,-0.141590,0.107867,0.135506,tool


# Metaphor Listing

In [3]:
# Myra/Cheng metaphors in table order (for reference)

MYRA_METAPHORS_IN_ORDER = [
    "tool",
    "brain",
    "search_engine",
    "assistant",
    "robot",
    "computer",
    "library",
    "future_shaper",
    "genie",
    "mirror",
    "child",
    "synthesizer",
    "teacher",
    "friend",
    "lifeform",
    "animal",
    "unexplored_realm",
    "god",
    "folklore",
    "thief",
]

# FrameScope metaphors (your schema)

FRAMESCOPE_METAPHORS = [
    "Tool",
    "Weapon",
    "Mirror",
    "Garbage",
    "Black Box",
    "Mind",
    "Friend",
    "Child",
    "Criminal",
    "Artist",
    "Assistant",
    "Animal",
    "Disaster",
    "Momentum",
    "Disease",
    "God",
    "Genie",
    "Oracle",
]

# STRICT INTERSECTION ONLY (no mapping)

EVALUATION_METAPHORS = [
    "Tool",
    "Assistant",
    "Genie",
    "Mirror",
    "Child",
    "Friend",
    "Animal",
    "God",
]

print("Evaluation metaphors:", EVALUATION_METAPHORS)
print("Count:", len(EVALUATION_METAPHORS))

Evaluation metaphors: ['Tool', 'Assistant', 'Genie', 'Mirror', 'Child', 'Friend', 'Animal', 'God']
Count: 8


# Filtering the dataset which is intersection of the our metaphor and myras

In [4]:
# Ensure consistent formatting
df["dominant_metaphor_clean"] = (
    df["dominant_metaphor"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# Create lowercase version of evaluation metaphors
EVAL_LOWER = [m.lower() for m in EVALUATION_METAPHORS]

# Filter dataset (STRICT intersection only)
df_filtered = df[df["dominant_metaphor_clean"].isin(EVAL_LOWER)].copy()

# Reset index for clean iteration later
df_filtered = df_filtered.reset_index(drop=True)

# Basic diagnostics
print("Original dataset size:", len(df))
print("Filtered dataset size:", len(df_filtered))
print("\nRemaining class distribution:\n")
print(df_filtered["dominant_metaphor_clean"].value_counts())

Original dataset size: 11789
Filtered dataset size: 4147

Remaining class distribution:

dominant_metaphor_clean
tool         1233
assistant     941
genie         498
mirror        419
child         419
friend        286
animal        210
god           141
Name: count, dtype: int64


In [5]:
OLLAMA_MODELS = [
    # Core LLaMA family
    "llama3.1:8b",
    "llama3.1:70b",    
    "llama3.2",

    # Strong open alternatives
    "mistral",
    "mixtral",           # mixture-of-experts → often better classification
    "gemma:7b",          # Google model (different bias)
    "phi3",              # small but surprisingly strong

    # Optional (only if you want more diversity)
    "qwen2",             # Alibaba model (good reasoning)
]

GPT_MODELS = [
    "gpt-5-nano",
    "gpt-4.1-nano",

    # Cheap but usable
    "gpt-5-mini",
    "gpt-4.1-mini",
    "gpt-4o-mini",

    # Mid-tier (good reasoning vs cost)
    "gpt-5",
    "gpt-4o",
    "o4-mini",

    # Reasoning-oriented
    "o3-mini",

    "gpt-4.1",
]

## LLAMA


In [6]:
import time
import requests
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# =========================
# CONFIG
# =========================

OLLAMA_URL = "http://localhost:11434/api/generate"

LLM_CONFIG = {
    "temperature": 0.0,
    "top_p": 1.0,
    "top_k": 40,
    "repeat_penalty": 1.0,
    "num_predict": 20  # short output since only 1 word
}

MAX_WORKERS = 4   # increase to 6–8 if stable
REQUEST_TIMEOUT = 120


# =========================
# LOAD PROMPT
# =========================

with open("../prompts/metaphor_classification_v1.txt", "r") as f:
    PROMPT_TEMPLATE = f.read()

def build_prompt(text):
    return PROMPT_TEMPLATE.replace("{input_text}", text)


# =========================
# CLEAN OUTPUT
# =========================

VALID_LABELS = [
    "Tool", "Assistant", "Genie", "Mirror",
    "Child", "Friend", "Animal", "God", "None"
]

def clean_prediction(pred):
    if not isinstance(pred, str):
        return "None"

    pred = pred.strip().replace('"', '')
    pred = pred.split("\n")[0]  # remove extra lines
    pred = pred.replace(".", "")
    pred = pred.capitalize()

    if pred not in VALID_LABELS:
        return "None"

    return pred


# =========================
# SINGLE CALL
# =========================

def call_ollama(prompt, model):
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": LLM_CONFIG
    }

    start = time.time()

    response = requests.post(
        OLLAMA_URL,
        json=payload,
        timeout=REQUEST_TIMEOUT
    )
    response.raise_for_status()

    raw = response.json()["response"]

    latency = time.time() - start

    pred = clean_prediction(raw)

    return pred, latency


# =========================
# WORKER FUNCTION
# =========================

def process_row(row, model):
    text = row["ai_metaphor"]
    reference = row["dominant_metaphor"]

    prompt = build_prompt(text)

    try:
        pred, latency = call_ollama(prompt, model)
        error = None
    except Exception as e:
        pred = "ERROR"
        latency = None
        error = str(e)

    return {
        "model_name": model,
        "ai_text": text,
        "reference_metaphor": reference,
        "predicted_metaphor": pred,
        "latency_seconds": latency,
        "cost_usd": 0.0,
        "error": error
    }


# =========================
# PARALLEL EXECUTION
# =========================

def run_llama_parallel(df, models):
    all_results = []

    for model in models:
        print(f"\nRunning model: {model}")

        results = []

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = [
                executor.submit(process_row, row, model)
                for _, row in df.iterrows()
            ]

            for f in tqdm(as_completed(futures), total=len(futures)):
                results.append(f.result())

        all_results.extend(results)

    return pd.DataFrame(all_results)

/Volumes/SSD500GB/FrameScope/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [ ]:
llama_results_df = run_llama_parallel(df_filtered, OLLAMA_MODELS)


Running model: llama3.1:8b


100%|██████████| 4147/4147 [17:07<00:00,  4.03it/s]



Running model: llama3.1:70b


100%|██████████| 4147/4147 [00:04<00:00, 938.84it/s]



Running model: llama3.2


100%|██████████| 4147/4147 [07:57<00:00,  8.69it/s]



Running model: mistral


 49%|████▉     | 2042/4147 [09:18<08:50,  3.97it/s]